In [1]:
import torch
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import pyproj

torch.set_printoptions(sci_mode = False)

# Data

- Bedmap3 UTIG 2009 ICECAP
    - minor confusion about year: 2009 or 2010
- Download [link ramadda](https://ramadda.data.bas.ac.uk/repository/entry/get/UTIG_2010_ICECAP_AIR_BM3.csv?entryid=synth%3A91523ff9-d621-46b3-87f7-ffb6efcd1847%3AL1VUSUdfMjAxMF9JQ0VDQVBfQUlSX0JNMy5jc3Y%3D
)
- 9 M data points

In [62]:
# Load data
utig_byrd = pd.read_csv("~/data/bedmap/UTIG_2010_ICECAP_AIR_BM3.csv", skiprows = range(0, 18))

# Find the right trajectory

In [70]:
# Subset for the coordinates we are interested in (by longitude)
# 30 seconds are 0.5?!
utig_byrd_subset = utig_byrd[utig_byrd["longitude (degree_east)"] >= 161.5][utig_byrd["longitude (degree_east)"] <= 162]

# plot histo of number of data points as we want flightlines roughly parallel to longitude lines so that it cross-sects the Byrd glacier
fig = px.histogram(x = utig_byrd_subset["trajectory_id"])
fig.show()

/tmp/ipykernel_38304/2565698604.py:3: UserWarning:

Boolean Series key will be reindexed to match DataFrame index.



In [71]:
np.max(utig_byrd_subset["trajectory_id"])

'IR2HI2_2012346_WSB_JKB2h_R40b'

In [72]:
# Calculate range of latitude and order
pd.DataFrame(utig_byrd_subset.groupby(by = ["trajectory_id"]).max(["latitude (degree_north)"])["latitude (degree_north)"]-utig_byrd_subset.groupby(by = ["trajectory_id"]).min(["latitude (degree_north)"])["latitude (degree_north)"]).sort_values(by = "latitude (degree_north)", ascending = False)

,latitude (degree_north)
trajectory_id,
IR2HI2_2011325_DVG_JKB2e_Y26a,0.665113
IR1HI2_2010341_WSB_JKB1a_GL0262a,0.399564
IR1HI2_2010342_WSB_JKB1a_GL0143a,0.374129
IR1HI2_2010333_WSB_JKB1a_GL0024b,0.335654
IR2HI2_2011333_WSB_JKB2e_SKEG01a,0.293216
...,...
IR2HI2_2011322_WSB_JKB2e_BYRD07b,0.010855
IR1HI2_2009342_WSB_JKB1a_R42a,0.010505
IR1HI2_2009336_WSB_JKB1a_BCN02e,0.010185


In [81]:
flight = utig_byrd_subset[utig_byrd_subset["trajectory_id"] == 'IR2HI2_2011325_DVG_JKB2e_Y26a']
# cut out turn around
flight = flight[flight["latitude (degree_north)"] < -74.84]
# Remove missing values
flight = flight[flight["bedrock_altitude (m)"] > -9000]
flight

,trajectory_id,trace_number,longitude (degree_east),latitude (degree_north),date,time_UTC,surface_altitude (m),land_ice_thickness (m),bedrock_altitude (m),two_way_travel_time (m),aircraft_altitude (m),along_track_distance (m)
7964107,IR2HI2_2011325_DVG_JKB2e_Y26a,-9999,161.500396,-75.470424,2011-11-21,02:21:44,612.78,177.28,435.51,-9999,-9999,-9999
7964108,IR2HI2_2011325_DVG_JKB2e_Y26a,-9999,161.500516,-75.470223,2011-11-21,02:21:45,612.67,180.63,432.04,-9999,-9999,-9999
7964111,IR2HI2_2011325_DVG_JKB2e_Y26a,-9999,161.500880,-75.469613,2011-11-21,02:21:45,612.25,179.76,432.49,-9999,-9999,-9999
7964112,IR2HI2_2011325_DVG_JKB2e_Y26a,-9999,161.501001,-75.469411,2011-11-21,02:21:46,612.18,181.07,431.11,-9999,-9999,-9999
7964114,IR2HI2_2011325_DVG_JKB2e_Y26a,-9999,161.501246,-75.469003,2011-11-21,02:21:46,612.65,183.67,428.98,-9999,-9999,-9999
...,...,...,...,...,...,...,...,...,...,...,...,...
7967186,IR2HI2_2011325_DVG_JKB2e_Y26a,-9999,161.848345,-74.841079,2011-11-21,02:34:59,1035.08,109.44,925.64,-9999,-9999,-9999
7967187,IR2HI2_2011325_DVG_JKB2e_Y26a,-9999,161.848456,-74.840875,2011-11-21,02:34:59,1035.21,108.56,926.66,-9999,-9999,-9999
7967189,IR2HI2_2011325_DVG_JKB2e_Y26a,-9999,161.848676,-74.840469,2011-11-21,02:34:59,1036.27,109.12,927.15,-9999,-9999,-9999
7967190,IR2HI2_2011325_DVG_JKB2e_Y26a,-9999,161.848787,-74.840259,2011-11-21,02:35:00,1035.97,107.47,928.51,-9999,-9999,-9999


In [82]:
print(np.max(flight["latitude (degree_north)"]))
print(np.min(flight["latitude (degree_north)"]))

-74.840055
-75.470424


# Visualise

In [83]:
fig = go.Figure(data = go.Scatter(x = flight[flight["trajectory_id"] == 'IR2HI2_2011325_DVG_JKB2e_Y26a']["longitude (degree_east)"], 
                                  y = flight[flight["trajectory_id"] == 'IR2HI2_2011325_DVG_JKB2e_Y26a']["latitude (degree_north)"], 
                                  mode = 'markers'))
fig.update_layout(title = "Flight line 11", template = "plotly_white")
# Dome A is around 1 M
fig.show()

In [96]:
fig = go.Figure(data = go.Scatter(x = flight["latitude (degree_north)"], 
                                  y = flight["bedrock_altitude (m)"], 
                                  mode = 'lines'))

fig.add_trace(go.Scatter(x = flight["latitude (degree_north)"], 
                         y = flight["surface_altitude (m)"] + 200, # artifically elevate
                         mode = 'lines'))

fig.add_trace(go.Scatter(x = flight["latitude (degree_north)"], 
                         y = flight["surface_altitude (m)"], # artifically elevate
                         mode = 'lines'))

fig.update_layout(title = "Flight line 11", template = "plotly_white")
# Dome A is around 1 M
fig.show()

# Convert to meters

Meaningless:
- along_track_distance (m)
- aircraft_altitude (m)
- two_way_travel_time (m)
- trace_number

In [106]:
flight

,trajectory_id,trace_number,longitude (degree_east),latitude (degree_north),date,time_UTC,surface_altitude (m),land_ice_thickness (m),bedrock_altitude (m),two_way_travel_time (m),aircraft_altitude (m),along_track_distance (m)
7964107,IR2HI2_2011325_DVG_JKB2e_Y26a,-9999,161.500396,-75.470424,2011-11-21,02:21:44,612.78,177.28,435.51,-9999,-9999,-9999
7964108,IR2HI2_2011325_DVG_JKB2e_Y26a,-9999,161.500516,-75.470223,2011-11-21,02:21:45,612.67,180.63,432.04,-9999,-9999,-9999
7964111,IR2HI2_2011325_DVG_JKB2e_Y26a,-9999,161.500880,-75.469613,2011-11-21,02:21:45,612.25,179.76,432.49,-9999,-9999,-9999
7964112,IR2HI2_2011325_DVG_JKB2e_Y26a,-9999,161.501001,-75.469411,2011-11-21,02:21:46,612.18,181.07,431.11,-9999,-9999,-9999
7964114,IR2HI2_2011325_DVG_JKB2e_Y26a,-9999,161.501246,-75.469003,2011-11-21,02:21:46,612.65,183.67,428.98,-9999,-9999,-9999
...,...,...,...,...,...,...,...,...,...,...,...,...
7967186,IR2HI2_2011325_DVG_JKB2e_Y26a,-9999,161.848345,-74.841079,2011-11-21,02:34:59,1035.08,109.44,925.64,-9999,-9999,-9999
7967187,IR2HI2_2011325_DVG_JKB2e_Y26a,-9999,161.848456,-74.840875,2011-11-21,02:34:59,1035.21,108.56,926.66,-9999,-9999,-9999
7967189,IR2HI2_2011325_DVG_JKB2e_Y26a,-9999,161.848676,-74.840469,2011-11-21,02:34:59,1036.27,109.12,927.15,-9999,-9999,-9999
7967190,IR2HI2_2011325_DVG_JKB2e_Y26a,-9999,161.848787,-74.840259,2011-11-21,02:35:00,1035.97,107.47,928.51,-9999,-9999,-9999


In [110]:
flight_sub = flight[["trajectory_id", "latitude (degree_north)", "longitude (degree_east)", "surface_altitude (m)", "bedrock_altitude (m)", "land_ice_thickness (m)"]]
flight_sub.columns = ["trajectory_id", "lat", "lon", "s", "b", "h"]

flight_sub

,trajectory_id,lat,lon,s,b,h
7964107,IR2HI2_2011325_DVG_JKB2e_Y26a,-75.470424,161.500396,612.78,435.51,177.28
7964108,IR2HI2_2011325_DVG_JKB2e_Y26a,-75.470223,161.500516,612.67,432.04,180.63
7964111,IR2HI2_2011325_DVG_JKB2e_Y26a,-75.469613,161.500880,612.25,432.49,179.76
7964112,IR2HI2_2011325_DVG_JKB2e_Y26a,-75.469411,161.501001,612.18,431.11,181.07
7964114,IR2HI2_2011325_DVG_JKB2e_Y26a,-75.469003,161.501246,612.65,428.98,183.67
...,...,...,...,...,...,...
7967186,IR2HI2_2011325_DVG_JKB2e_Y26a,-74.841079,161.848345,1035.08,925.64,109.44
7967187,IR2HI2_2011325_DVG_JKB2e_Y26a,-74.840875,161.848456,1035.21,926.66,108.56
7967189,IR2HI2_2011325_DVG_JKB2e_Y26a,-74.840469,161.848676,1036.27,927.15,109.12
7967190,IR2HI2_2011325_DVG_JKB2e_Y26a,-74.840259,161.848787,1035.97,928.51,107.47


In [112]:
# This is how projections work. check through tool
polarstereo_to_lonlat = pyproj.Transformer.from_crs(crs_from = pyproj.CRS("epsg:4326"),  
                                                    crs_to = pyproj.CRS("epsg:3031"),
                                                    always_xy = True) # xy order convention

# Pass in lon, lat and return x, y in polar stereo
x_array, y_array = polarstereo_to_lonlat.transform(
    flight_sub["lon"], 
    flight_sub["lat"])

flight_sub["x"] = x_array
flight_sub["y"] = y_array

flight_sub

/tmp/ipykernel_38304/2182446399.py:11: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/ipykernel_38304/2182446399.py:12: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



,trajectory_id,lat,lon,s,b,h,x,y
7964107,IR2HI2_2011325_DVG_JKB2e_Y26a,-75.470424,161.500396,612.78,435.51,177.28,503503.449852,-1.504848e+06
7964108,IR2HI2_2011325_DVG_JKB2e_Y26a,-75.470223,161.500516,612.67,432.04,180.63,503507.335688,-1.504870e+06
7964111,IR2HI2_2011325_DVG_JKB2e_Y26a,-75.469613,161.500880,612.25,432.49,179.76,503519.132692,-1.504937e+06
7964112,IR2HI2_2011325_DVG_JKB2e_Y26a,-75.469411,161.501001,612.18,431.11,181.07,503523.026921,-1.504959e+06
7964114,IR2HI2_2011325_DVG_JKB2e_Y26a,-75.469003,161.501246,612.65,428.98,183.67,503530.876358,-1.505004e+06
...,...,...,...,...,...,...,...,...
7967186,IR2HI2_2011325_DVG_JKB2e_Y26a,-74.841079,161.848345,1035.08,925.64,109.44,516005.219486,-1.573913e+06
7967187,IR2HI2_2011325_DVG_JKB2e_Y26a,-74.840875,161.848456,1035.21,926.66,108.56,516009.192822,-1.573936e+06
7967189,IR2HI2_2011325_DVG_JKB2e_Y26a,-74.840469,161.848676,1036.27,927.15,109.12,516017.125343,-1.573980e+06
7967190,IR2HI2_2011325_DVG_JKB2e_Y26a,-74.840259,161.848787,1035.97,928.51,107.47,516021.304976,-1.574004e+06


In [125]:
flight_sub["y"].astype(str)

selection = flight_sub[flight_sub["y"] > (-1505000 - 10000)]

-1514814.9999644863

In [126]:
np.max(selection["y"])

-1504847.7538263432

min -1514815
max -1504847.7

In [132]:
fig = go.Figure(data = go.Scatter(x = selection["y"] - -1514815, 
                                  y = selection["b"], 
                                  mode = 'lines'))

fig.add_trace(go.Scatter(x = selection["y"] - -1514815, 
                         y = selection["s"], # artifically elevate
                         mode = 'lines'))
fig.add_hline(y = 0)

fig.update_layout(template = "plotly_white")
fig.update_layout(yaxis_range = [- 20, 660])
fig.update_layout(xaxis_range = [2, 10000])

fig.show()